In [ ]:
# __ri_prediction_notebook_setup__
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_DIR = PROJECT_ROOT / "data"
TRAINING_DATA_DIR = DATA_DIR / "training"
CASE_DATA_DIR = DATA_DIR / "cases"
EXAMPLE_DATA_DIR = DATA_DIR / "examples"
MODEL_DIR = PROJECT_ROOT / "models" / "pretrained"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)


In [ ]:
import pandas as pd
import math
import numpy as np
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix, classification_report
from sklearn import preprocessing, svm, neighbors
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt


In [ ]:
df = pd.read_csv(TRAINING_DATA_DIR / "SHIP_AtlanticMajorHurricane2016-2019.csv")
print(df.head(10))
df.replace('?',-99999, inplace=True)
df.drop(['OHC','LAND'], axis=1, inplace=True)
x = np.array(df.drop(['class'],axis=1)).astype("float32")
y = np.array(df['class']).astype("float32")
print(x[0],y[0])

In [ ]:
import copy
random_label_copy = copy.copy(y)
print(random_label_copy[0:50])
np.random.shuffle(random_label_copy)
print(random_label_copy[0:50])
hits_array = np.array(y) == np.array(random_label_copy)
#print(hits_array)
print(hits_array.mean())

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

In [ ]:
x1_train, x1_test, y1_train, y1_test = train_test_split(x, y, test_size=0.2)
print(x1_train.shape,x1_test.shape,y1_train.shape,y1_test.shape)
validation_split = np.int(0.14*len(x1_train))
#print("validation split gives",validation_split)
x2_val = x1_train[:validation_split]
y2_val = y1_train[:validation_split]
x2_train = x1_train[validation_split:]
y2_train = y1_train[validation_split:]
#print(y2_train)
print(x2_train.shape,x2_val.shape,y2_train.shape,y2_val.shape)

model = keras.Sequential([
    layers.Dense(32, activation = "relu"),
    layers.Dense(1, activation = "sigmoid")])

model.compile(optimizer="rmsprop",loss="binary_crossentropy",metrics=["accuracy"])

history = model.fit(x2_train,y2_train,epochs=50,batch_size=16,validation_data=(x2_val,y2_val))

In [ ]:
import matplotlib.pyplot as plt
#print(history.__dict__)
epochs = history.epoch
val_loss = history.history['val_loss']
loss = history.history['loss']
#print(loss)
plt.plot(epochs,val_loss,'r+',label="Validation loss")
plt.xlabel("epochs")
plt.ylabel("value")
plt.title("Loss history")
plt.plot(epochs,loss,'b',label="Training loss")
plt.legend()
plt.show()

plt.clf()
val_accuracy = history.history['val_accuracy']
accuracy = history.history['accuracy']
#print(loss)
plt.plot(epochs,val_accuracy,'r+',label="Validation accuracy")
plt.xlabel("epochs")
plt.ylabel("value")
plt.title("Accuracy history")
plt.plot(epochs,accuracy,'b',label="Accuracy")
plt.legend()
plt.show()


In [ ]:
results = model.evaluate(x1_test,y1_test)
print("Evalution results (loss,accuracy) for the test data is ",results)
single_fcst = model.predict(x1_test)
for i in range(len(single_fcst)):
    print(y1_test[i]," <---> RI Propbability: ",f"{float(single_fcst[i]):.5f}")
